In [ ]:
import pyarrow.compute as pc
import pyarrow.parquet as pq

OUT = "../data/processed/simplewiki"
pages = pq.read_table(f"{OUT}/pages.parquet")
edges = pq.read_table(f"{OUT}/edges.parquet")
print(pages.schema)
print(edges.schema)
print(pages.num_rows, edges.num_rows)

ids = pages["page_id"]
print("all sources are articles:", pc.all(pc.is_in(edges["source"], value_set=ids)).as_py())
print("all targets are articles:", pc.all(pc.is_in(edges["target"], value_set=ids)).as_py())

id_to_title = dict(zip(pages["page_id"].to_pylist(), pages["title"].to_pylist(), strict=False))
title_to_id = {title: page_id for page_id, title in id_to_title.items()}

munich = title_to_id["Munich"]
out_links = edges.filter(pc.equal(edges["source"], munich))["target"].to_pylist()
print(len(out_links), "links from Munich, e.g.:")
print([id_to_title[t] for t in out_links[:15]])